In [1]:
import torch
import numpy as np

from ripser import ripser

In [3]:
representations_path = "resnet_epoch1_vectorized_representations.pt"

representations = torch.load(
    representations_path,
    map_location="cpu"
)

print("Number of blocks:", len(representations))

for i, rep in enumerate(representations):
    print(f"Block {i+1}: {rep.shape}")

Number of blocks: 10
Block 1: torch.Size([1000, 128])
Block 2: torch.Size([1000, 128])
Block 3: torch.Size([1000, 128])
Block 4: torch.Size([1000, 128])
Block 5: torch.Size([1000, 128])
Block 6: torch.Size([1000, 128])
Block 7: torch.Size([1000, 128])
Block 8: torch.Size([1000, 128])
Block 9: torch.Size([1000, 128])
Block 10: torch.Size([1000, 128])


In [5]:
sample_size = 300
random_seed = 42

generator = torch.Generator().manual_seed(random_seed)

indices = torch.randperm(
    representations[0].shape[0],
    generator=generator
)[:sample_size]

print("Number of sampled points:", len(indices))
print("First 10 indices:", indices[:10])

Number of sampled points: 300
First 10 indices: tensor([542, 618, 816,  68,  94, 215,  60, 585, 942, 165])


In [7]:
for i, rep in enumerate(representations):
    sample = rep[indices]

    print(
        f"Block {i+1}: "
        f"{sample.shape}"
    )

Block 1: torch.Size([300, 128])
Block 2: torch.Size([300, 128])
Block 3: torch.Size([300, 128])
Block 4: torch.Size([300, 128])
Block 5: torch.Size([300, 128])
Block 6: torch.Size([300, 128])
Block 7: torch.Size([300, 128])
Block 8: torch.Size([300, 128])
Block 9: torch.Size([300, 128])
Block 10: torch.Size([300, 128])


In [9]:
def lifespan_sum(diagram, alpha=1.0):
    """
    Compute the power-weighted sum of finite persistence lifespans.

    E_alpha = sum((death - birth)^alpha)
    """

    finite_mask = np.isfinite(diagram[:, 1])

    finite_pairs = diagram[finite_mask]

    if len(finite_pairs) == 0:
        return 0.0

    lifespans = (
        finite_pairs[:, 1] -
        finite_pairs[:, 0]
    )

    return np.sum(lifespans ** alpha)

In [13]:
# Test the lifespan_sum function

test_diagram = np.array([
    [0.2, 0.5],
    [0.4, 0.9],
    [0.7, 1.0]
])

test_value = lifespan_sum(
    test_diagram,
    alpha=1
)

print("Test descriptor:", test_value)

Test descriptor: 1.1


In [15]:
h0_descriptors = []
h1_descriptors = []

for block_idx, representation in enumerate(representations):

    print(f"Processing Block {block_idx + 1}/10...")

    sample = representation[indices].numpy()

    result = ripser(
        sample,
        maxdim=1
    )

    h0_diagram = result["dgms"][0]
    h1_diagram = result["dgms"][1]

    h0_value = lifespan_sum(
        h0_diagram,
        alpha=1
    )

    h1_value = lifespan_sum(
        h1_diagram,
        alpha=1
    )

    h0_descriptors.append(h0_value)
    h1_descriptors.append(h1_value)

    print(
        f"  H0 descriptor: {h0_value:.4f}"
    )

    print(
        f"  H1 descriptor: {h1_value:.4f}"
    )

Processing Block 1/10...
  H0 descriptor: 341.8065
  H1 descriptor: 16.1193
Processing Block 2/10...
  H0 descriptor: 498.1971
  H1 descriptor: 30.0796
Processing Block 3/10...
  H0 descriptor: 659.7802
  H1 descriptor: 46.4136
Processing Block 4/10...
  H0 descriptor: 771.6399
  H1 descriptor: 50.1310
Processing Block 5/10...
  H0 descriptor: 900.1313
  H1 descriptor: 62.1575
Processing Block 6/10...
  H0 descriptor: 1020.9185
  H1 descriptor: 71.1731
Processing Block 7/10...
  H0 descriptor: 1143.3094
  H1 descriptor: 79.6560
Processing Block 8/10...
  H0 descriptor: 1265.5297
  H1 descriptor: 83.5401
Processing Block 9/10...
  H0 descriptor: 1354.5738
  H1 descriptor: 84.1724
Processing Block 10/10...
  H0 descriptor: 1411.6602
  H1 descriptor: 84.4449


In [17]:
h0_descriptors = np.array(h0_descriptors)
h1_descriptors = np.array(h1_descriptors)

print("H0 descriptors:")
print(h0_descriptors)

print("\nH1 descriptors:")
print(h1_descriptors)

H0 descriptors:
[ 341.80649096  498.19714296  659.78021371  771.63988531  900.13132465
 1020.91854298 1143.30939889 1265.52969313 1354.57382214 1411.66024935]

H1 descriptors:
[16.11929268 30.07961977 46.41364968 50.1310215  62.15753436 71.17309332
 79.65604091 83.54013467 84.17242312 84.44485354]


In [19]:
print("H0 NaN:", np.isnan(h0_descriptors).any())
print("H0 Inf:", np.isinf(h0_descriptors).any())

print("H1 NaN:", np.isnan(h1_descriptors).any())
print("H1 Inf:", np.isinf(h1_descriptors).any())

print("Number of H0 descriptors:", len(h0_descriptors))
print("Number of H1 descriptors:", len(h1_descriptors))

H0 NaN: False
H0 Inf: False
H1 NaN: False
H1 Inf: False
Number of H0 descriptors: 10
Number of H1 descriptors: 10


In [21]:
print("Block-wise topological descriptors")
print("-----------------------------------")

for i in range(10):

    print(
        f"Block {i+1:2d} | "
        f"H0 = {h0_descriptors[i]:.4f} | "
        f"H1 = {h1_descriptors[i]:.4f}"
    )

Block-wise topological descriptors
-----------------------------------
Block  1 | H0 = 341.8065 | H1 = 16.1193
Block  2 | H0 = 498.1971 | H1 = 30.0796
Block  3 | H0 = 659.7802 | H1 = 46.4136
Block  4 | H0 = 771.6399 | H1 = 50.1310
Block  5 | H0 = 900.1313 | H1 = 62.1575
Block  6 | H0 = 1020.9185 | H1 = 71.1731
Block  7 | H0 = 1143.3094 | H1 = 79.6560
Block  8 | H0 = 1265.5297 | H1 = 83.5401
Block  9 | H0 = 1354.5738 | H1 = 84.1724
Block 10 | H0 = 1411.6602 | H1 = 84.4449


In [23]:
descriptor_data = {
    "h0_descriptors": torch.tensor(h0_descriptors),
    "h1_descriptors": torch.tensor(h1_descriptors),
    "sample_indices": indices
}

torch.save(
    descriptor_data,
    "resnet_epoch1_topological_descriptors.pt"
)

print("Saved:")
print("resnet_epoch1_topological_descriptors.pt")

Saved:
resnet_epoch1_topological_descriptors.pt
